# Notebook 01 — Data Loading & Cleaning
**Dataset:** DataCo Supply Chain (Kaggle, ~180k transaksi)  
**Catatan:** File CSV menggunakan encoding `latin-1`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path('../data/raw')
OUT = Path('../output')
FIGURES = OUT / 'figures'
OUT.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

## 1. Load Data

In [ ]:
df = pd.read_csv(RAW / 'DataCoSupplyChainDataset.csv', encoding='latin-1')
print(f'Shape: {df.shape}')
df.head(2)

## 2. Rename Kolom (bersihkan spasi & karakter khusus)

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
    .str.replace('/', '_', regex=False)
)
print('Kolom setelah rename:')
print(df.columns.tolist())

## 3. Parse Tanggal & Feature Engineering

In [ ]:
# Cari kolom tanggal
date_cols = [c for c in df.columns if 'date' in c.lower()]
print('Kolom tanggal:', date_cols)

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Identifikasi kolom lead time
ship_real = [c for c in df.columns if 'shipping' in c and 'real' in c]
ship_sched = [c for c in df.columns if 'shipment' in c and 'scheduled' in c]
print('Lead time real    :', ship_real)
print('Lead time sched   :', ship_sched)

In [ ]:
# Nama kolom standar berdasarkan dataset DataCo
lead_real  = 'days_for_shipping_real'
lead_sched = 'days_for_shipment_scheduled'
deliv_col  = 'delivery_status'

# Fallback jika nama sedikit berbeda
if lead_real not in df.columns:
    lead_real = [c for c in df.columns if 'shipping' in c and 'real' in c][0]
if lead_sched not in df.columns:
    lead_sched = [c for c in df.columns if 'shipment' in c and 'scheduled' in c][0]

df['actual_lead_time']    = df[lead_real]
df['scheduled_lead_time'] = df[lead_sched]
df['delay_days']          = df['actual_lead_time'] - df['scheduled_lead_time']
df['is_late']             = df[deliv_col].str.contains('Late', case=False, na=False).astype(int)

# Bulan order
order_date_col = [c for c in df.columns if 'order_date' in c or 'dateorders' in c]
if order_date_col:
    df['order_year_month'] = df[order_date_col[0]].dt.to_period('M')

print('Feature engineering selesai.')
print(f"Late delivery rate: {df['is_late'].mean():.1%}")

## 4. Drop Kolom Tidak Relevan

In [ ]:
drop_keywords = ['image', 'description', 'latitude', 'longitude', 'zipcode', 'password']
drop_cols = [c for c in df.columns
             if any(kw in c.lower() for kw in drop_keywords)]
df = df.drop(columns=drop_cols)
print(f'Drop {len(drop_cols)} kolom: {drop_cols}')
print(f'Shape setelah drop: {df.shape}')

## 5. Cek Missing Values

In [ ]:
missing = df.isnull().sum()
print('Kolom dengan missing values:')
print(missing[missing > 0].sort_values(ascending=False))

# Drop baris dengan missing di kolom kunci
key_cols = ['actual_lead_time', 'scheduled_lead_time', 'is_late']
df = df.dropna(subset=key_cols)
print(f'\nShape final: {df.shape}')

## 6. Ringkasan Dataset

In [ ]:
print('=== RINGKASAN DATASET ===')
print(f'Total transaksi       : {len(df):,}')
print(f'Produk unik           : {df["product_name"].nunique():,}')
print(f'Kategori unik         : {df["category_name"].nunique():,}')
print(f'Market                : {df["market"].unique().tolist()}')
print(f'Shipping modes        : {df["shipping_mode"].unique().tolist()}')
print(f'Avg actual lead time  : {df["actual_lead_time"].mean():.1f} hari')
print(f'Late delivery rate    : {df["is_late"].mean():.1%}')

## 7. Simpan ke Parquet

In [ ]:
df.to_parquet(OUT / 'df_clean.parquet', index=False)
print(f'Tersimpan: df_clean.parquet — {len(df):,} baris, {df.shape[1]} kolom')